# Example 4.1 / 1: Combining Groups

Companion notebook for **4.1 Set Operations** in the AMPL lab Documentation, section "Combining Groups". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Combining Groups

The model receives the products that need minimum and maximum production levels. It derives
`PRODUCTS`, `BOTH`, and `ONE_SIDED`, avoiding a redundant master list in the data. In the Python
tabs, `ampl` is an `AMPL` object that has already read `model.mod`; `to_list()` retrieves each
computed set.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set MIN_REQUIRED;
set MAX_ALLOWED;

set PRODUCTS = MIN_REQUIRED union MAX_ALLOWED;
set BOTH = MIN_REQUIRED inter MAX_ALLOWED;
set MIN_ONLY = MIN_REQUIRED diff MAX_ALLOWED;
set ONE_SIDED = MIN_REQUIRED symdiff MAX_ALLOWED;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["MIN_REQUIRED"] = ["bands", "coils"]
ampl.set["MAX_ALLOWED"] = ["coils", "plate"]

In [5]:
# run.py
for name in ["PRODUCTS", "BOTH", "MIN_ONLY", "ONE_SIDED"]:
    print(name, sorted(ampl.set[name].to_list()))

PRODUCTS ['bands', 'coils', 'plate']
BOTH ['coils']
MIN_ONLY ['bands']
ONE_SIDED ['bands', 'plate']


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set MIN_REQUIRED := bands coils;
set MAX_ALLOWED := coils plate;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display PRODUCTS, BOTH, MIN_ONLY, ONE_SIDED;

set PRODUCTS := bands coils plate;

set BOTH := coils;

set MIN_ONLY := bands;

set ONE_SIDED := bands plate;

